In [ ]:
# 텍스트 정제 후 감성분류
samples = ["The cat say on the mat.", "The dog ate\t my\n homework"]    # 영문 ; 공백을 기준을 품사가 나뉨

# 수치화 1 : 단어 사전 생성 - 프로그래밍
import string
token_index = {} # Initialize token_index
for sam in samples:
  for word in sam.split():
    word = word.strip(string.punctuation).lower()
    # print(word)
    if word not in token_index:
      token_index[word] = len(token_index)

print(token_index)

print()
# 수치화 2 : 단어 사전 생성 - Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words = 10)
tokenizer.fit_on_texts(samples)
print(tokenizer.word_index)
token_seq = tokenizer.texts_to_sequences(samples)
print(token_seq)

token_mat = tokenizer.texts_to_matrix(samples, mode='binary')    # 'binary', 'count', 'freq', 'tfidf
print(token_mat)    # 0,1 로만 구성된 배열을 출력
print(tokenizer.word_counts)
print(tokenizer.document_count)
print(tokenizer.word_docs)

print()

# OneHotEncoding
from tensorflow.keras.utils import to_categorical
seq = token_seq[0]
num_classes = max(seq) + 1    # 첫 번째 인덱스가 0부터 시작해서 +1

token_seq = to_categorical(token_seq[0], num_classes=num_classes)
print(token_seq)

In [ ]:
print('영화 관람 후 평에 대한 선호 분류')
docs = ['너무 재밌네요', '최고예요', '참 잘 만든 영화예요','추천하고 싶은 영화입니다', '한 번 더 보고 싶네요',
        '글쎄요','별로예요','생각보다 지루하네요','연기가 어색해요','재미 없어요']

import numpy as np
labels = np.array([1,1,1,1,1,0,0,0,0,0])
token = Tokenizer()
token.fit_on_texts(docs)
print(token.word_index)    # 문장 인덱싱

x = token.texts_to_sequences(docs)
print(f"리뷰 토큰화 결과 : {x}")

# 시퀸스 데이터를 RNN 딥러닝 모델에 넣기 전에 길이를 맞추는 작업이 필요
from tensorflow.keras.utils import pad_sequences
padded_x = pad_sequences(x, maxlen=5, padding='pre')    # 'post':오른쪽 채우기, 'pre':왼쪽 채우기(기본)
print(f"패딩 결과 : {padded_x}")

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Input, Flatten

word_size = len(token.word_index) + 1
model = Sequential()
model.add(Input(shape=(5,)))
model.add(Embedding(input_dim=word_size, output_dim=8, input_length=5))
model.add(LSTM(32, activation='tanh'))
# return sequence가 True일때만 model.add(Flatten) 작성
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(padded_x, labels, epochs=30, batch_size=32, verbose=2)
print(f"정확도 : {model.evaluate(padded_x, labels)[1]:.4f}")

print(f"예측 확률 : {model.predict(padded_x)}")
print(f"예측 : {np.where(model.predict(padded_x)>0.5, 1, 0).ravel()}")